# Sports Arena: IPL Performance Analyzer (2008-2026)

## 1. Introduction

**Objective.** Use every ball of the Indian Premier League from **2008 to 2026** (1,243 matches, 295,729 deliveries)
to answer analyst questions, predict the **2027** season with two models, and check how far those predictions can be trusted.

**How this notebook works.** The project's code lives in `src/` (one job per file, every cricket formula in
`src/metrics.py`). Instead of copying that code here (and risking two versions), this notebook **downloads the
repository and imports the same functions**, then runs them step by step. Where it helps to read the code, a cell
prints the function's source with `inspect.getsource`.

**IPL background.** A T20 innings has at most 20 overs of 6 legal balls. Phases: Powerplay (overs 1-6),
Middle (7-15), Death (16-20). Since 2022 there are 10 teams; since 2023 each team may use an Impact Player substitute.

**Data.** Kaggle IPL 2008-2019 (CC BY-NC-SA 4.0) + Cricsheet 2020-2026 (credit: cricsheet.org; register under ODC-By 1.0),
merged into `data/merged/` and never edited. See `data/README.md`.

## 2. Setup

In Colab the first cell downloads the repository and installs the exact library versions from `requirements.txt`
(click *Restart session* if Colab asks, then **Runtime > Run all** again). Run inside the project's `notebooks/`
folder, it uses the local files instead.

In [ ]:
import os, sys, subprocess

# >>>>>>> EDIT IF NEEDED: the GitHub repository and branch <<<<<<<
GITHUB_USER = "Mohanp1821"
GITHUB_REPO = "sports-arena"
GITHUB_BRANCH = "main"

if os.path.exists("../src/metrics.py"):          # running inside the project's notebooks/ folder
    PROJECT = os.path.abspath("..")
else:                                            # running in Colab: download the project once
    PROJECT = os.path.abspath(GITHUB_REPO)
    if not os.path.exists(PROJECT):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
                        "https://github.com/" + GITHUB_USER + "/" + GITHUB_REPO + ".git", PROJECT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(PROJECT, "requirements.txt")], check=True)

sys.path.insert(0, os.path.join(PROJECT, "src"))
os.chdir(PROJECT)
print("Project folder:", PROJECT)

In [ ]:
import inspect
import pandas as pd
import matplotlib.pyplot as plt

import metrics, analysis, predict, predict_ml, dashboard_data, chat_facts   # the project's own files

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
analysis.SHOW_CHARTS = True       # show every chart here (they are also saved to outputs/)
analysis.setup_style()

## 3. Check and load the data

Step 1 of the pipeline: the SHA-256 checksums prove the data files are exactly the ones the project was built with.
The cleaned files in `data/processed/` were made by `src/prepare_data.py` (step 2) and are loaded here.

In [ ]:
import verify_data
verify_data.main()

matches, deliveries = metrics.load_processed_data()
impact = metrics.load_impact_players()
print("matches   :", matches.shape)
print("deliveries:", deliveries.shape)
print("Seasons   :", metrics.season_range_text(matches), "(" + str(matches["season"].nunique()) + " seasons)")
matches.head()

In [ ]:
deliveries.head()

## 4. The cleaning layer (what `prepare_data.py` adds and why)

| Rule | Problem | Fix |
|---|---|---|
| Franchises | Delhi Daredevils → Delhi Capitals, Kings XI Punjab → Punjab Kings, RCB Bangalore → Bengaluru | Keep the season's name for display; add a `..._franchise` column with today's name |
| Grounds | "Wankhede Stadium" vs "Wankhede Stadium, Mumbai"; renamed grounds | Remove ", City"; rename list (Kotla → Arun Jaitley, Motera → Narendra Modi, …) |
| Player names | Kaggle "S Gill" vs Cricsheet "Shubman Gill" | 71 fixes from `data/player_name_map.csv` (Cricsheet registry IDs + ball alignment) |
| Stage / phase | Playoffs and phases not marked | Last 3-4 matches of a season = playoffs; Powerplay / Middle / Death |

In [ ]:
# Season names are kept, the franchise column joins them.
pairs = matches[["team1", "team1_franchise"]].drop_duplicates()
pairs[pairs["team1"] != pairs["team1_franchise"]]

In [ ]:
# A few rows of the player-name map: (season, team, old name) -> new name, with the vote that decided it.
name_map = pd.read_csv("data/player_name_map.csv")
print(len(name_map), "fixes")
name_map[name_map["old_name"].isin(["S Gill", "J Archer", "Ankit Sharma", "AS Yadav", "NA Saini"])]

In [ ]:
# Shubman Gill now has one career, every season 2018-2026, under one name.
gill = metrics.batting_stats(deliveries, ["batter", "season"])
gill[gill["batter"] == "Shubman Gill"][["season", "runs", "strike_rate"]]

In [ ]:
# One name and one city per ground: Wankhede spans 2008-2026.
grounds = matches.groupby(["venue", "city"]).agg(matches=("match_id", "count"), first=("season", "min"), last=("season", "max"))
grounds.sort_values("matches", ascending=False).head(10)

## 5. The cricket formulas

Read the ball-level rules: a **wide** is not a ball faced; a legal ball is not a wide or no-ball; byes and leg-byes
are not the bowler's runs; a run out is not the bowler's wicket; super overs are removed.

In [ ]:
print(inspect.getsource(metrics.add_ball_columns))

In [ ]:
print(inspect.getsource(metrics.batting_stats))

In [ ]:
metrics.top_performers(deliveries, None, "runs")[["batter", "runs", "innings", "average", "strike_rate", "fifties", "hundreds"]]

In [ ]:
metrics.top_performers(deliveries, None, "wickets")[["bowler", "wickets", "overs", "economy", "bowling_avg", "dot_pct"]]

## 6. Player form and careers

In [ ]:
analysis.plot_player_form(deliveries, "V Kohli", 2016)
analysis.plot_player_career(deliveries, "V Kohli")
analysis.compare_players(deliveries, "V Kohli", "RG Sharma")

## 7. Team comparisons (by franchise)

In [ ]:
analysis.plot_alltime_win_pct(matches)
analysis.plot_bat_first_vs_chase(deliveries, matches)
analysis.plot_team_season_heatmap(matches)
toss_table, toss_overall = metrics.toss_impact(matches)
print("The toss winner won", toss_overall, "% of matches")

## 8. Orange and Purple Caps (checked against the official records in `tests/test_facts.py`)

In [ ]:
caps = metrics.cap_winners(deliveries)
analysis.plot_cap_winners(caps)
caps

## 9. Analyst views

### 9.1 Rivalry centre (change the two teams)

In [ ]:
TEAM_A, TEAM_B = "Chennai Super Kings", "Mumbai Indians"
overall, by_season, by_stage, by_ground = metrics.rivalry_record(matches, TEAM_A, TEAM_B)
print(overall)
display(by_stage)
display(metrics.rivalry_last_meetings(matches, TEAM_A, TEAM_B))
analysis.plot_rivalry(matches, TEAM_A, TEAM_B)

### 9.2 Batter vs bowler, and how a batter gets out

In [ ]:
display(metrics.batter_vs_bowler(deliveries, "V Kohli", "JJ Bumrah"))
display(metrics.dismissal_types(deliveries, "V Kohli"))
analysis.plot_batter_vs_bowlers(deliveries, "V Kohli")

### 9.3 Grounds and the home fortress index

In [ ]:
summary = metrics.ground_summary(deliveries, matches)
display(summary.sort_values("matches", ascending=False).head(8))
display(metrics.team_at_ground(matches, "Chennai Super Kings", "MA Chidambaram Stadium, Chepauk"))
analysis.plot_fortress(matches)

### 9.4 Pitch and player fit

There are **no pitch reports** in the data, so a ground's behaviour is measured from what happened there: runs, wickets,
boundaries and dot balls compared with the league in the **same seasons** (index 100 = average). A player is compared with
himself at other grounds in the same seasons.

In [ ]:
display(metrics.pitch_profile(deliveries, matches, min_matches=20)[["venue", "matches", "runs_index", "wickets_index",
                                                                     "boundary_index", "dot_index", "label"]])
analysis.plot_ground_map(deliveries, matches)

In [ ]:
fit = metrics.player_ground_batting(deliveries, matches)
display(fit[(fit["batter"] == "V Kohli") & (fit["balls"] >= 150)][["venue", "balls", "strike_rate", "else_strike_rate", "strike_rate_diff"]])
analysis.plot_player_ground_fit(deliveries, matches, "V Kohli")

### 9.5 Phase specialists, finishers and partnerships

In [ ]:
display(metrics.phase_batting_leaders(deliveries, "Death", min_balls=300))
display(metrics.phase_bowling_leaders(deliveries, "Powerplay", min_balls=300))
analysis.plot_finishers(deliveries, matches)
analysis.plot_partnerships(deliveries, matches)

### 9.6 The Impact Player era (2020-22 vs 2023-26)

In [ ]:
display(metrics.impact_era_summary(deliveries, matches))
analysis.plot_impact_era(deliveries, matches)
analysis.plot_impact_choices(impact, deliveries, matches)

### 9.7 Trends: scoring inflation, a rebuilt points table, chase win probability, impact scores

In [ ]:
analysis.plot_scoring_inflation(deliveries, matches)
metrics.points_table(deliveries, matches, 2026)

In [ ]:
# Logistic regression on every ball of every normal chase.
model = metrics.win_probability_model(metrics.chase_states(deliveries, matches))
print("60 needed off 60 balls, 7 wickets left:", metrics.win_probability(model, 60, 60, 7), "%")
analysis.plot_chase_curve(model)
analysis.plot_impact_scores(deliveries, int(matches["season"].max()))

## 10. Predicting 2027: Model A vs Model B

**Model A** (`src/predict.py`): strength = form win % (last 3 seasons, weights 3-2-1); team A beats team B with chance
A ÷ (A + B); the season is played 10,000 times in the real format (two groups of 5, 14 games each, then the playoffs).

**Model B** (`src/predict_ml.py`): logistic regression + gradient boosting on pre-season features (Elo, last-10 form,
head-to-head, ground record, home flag, squad strength, Impact Player era), averaged and fed into the same simulator.

Both read `data/squads_2027.csv` (edit it after the auction). The next cell runs both scripts exactly as the pipeline does
(about 30 seconds), including the walk-forward backtest 2021-2026.

In [ ]:
print(inspect.getsource(predict.weighted_form))

In [ ]:
analysis.SHOW_CHARTS = False      # the scripts save their charts; they are shown as images below
predict.main()
predict_ml.main()

In [ ]:
comparison = pd.read_csv("outputs/prediction_2027_comparison.csv")
comparison

In [ ]:
pd.read_csv("outputs/model_comparison_matches.csv").query("season == '2021-2026'")

In [ ]:
from IPython.display import Image, display
for chart in ["prediction_2027_models.png", "model_comparison_backtest.png", "model_calibration.png"]:
    display(Image("outputs/" + chart))

**Reading the backtest honestly.** Neither model beats a coin flip's log loss (0.693) when predicting matches before the
season starts, and both are over-confident. The models explain *who looks strong and why*; they cannot know auctions,
injuries or breakout players.

## 11. Ask Sports Arena (the chatbot)

`src/chat_facts.py` writes every fact the chatbot may use to `outputs/chat_facts.json`. The answer engine is plain
JavaScript (`src/chatbot.js`), the same code the dashboard runs. Colab has Node.js, so we can ask it questions here
(if Node.js is missing, these cells say so).

In [ ]:
facts = chat_facts.build_facts(matches, deliveries, impact)
import shutil, json

def ask(question):
    """Ask the chatbot (src/chatbot.js) a question through Node.js and print its answer and source."""
    if shutil.which("node") is None:
        print("Node.js is not installed here, so the chatbot cannot run in this notebook.")
        return
    script = ("const f=require('./outputs/chat_facts.json').facts; const c=require('./src/chatbot.js');"
              "const a=c.answerQuestion(" + json.dumps(question) + ", f); console.log(a.text); console.log('['+a.source+']');")
    print(subprocess.run(["node", "-e", script], capture_output=True, text=True).stdout)

for question in ["Who won the Orange Cap in 2016?", "Kohli vs Bumrah", "How does the pitch at Chepauk play?",
                 "Kohli at Chinnaswamy", "CSK vs MI head to head",
                 "Who will win IPL 2027?", "What is the weather tomorrow?"]:
    print("Q:", question)
    ask(question)

In [ ]:
if shutil.which("node"):
    print(subprocess.run(["node", "tests/test_chatbot.js"], capture_output=True, text=True).stdout[-300:])

## 12. Key insights (calculated, not typed)

In [ ]:
import build_report
for sentence in build_report.key_insights(matches, deliveries):
    print("-", sentence)

## 13. How to reproduce

* **Colab:** Runtime → Run all (this notebook).
* **Terminal:** `./run_all.sh` (8 steps; builds `outputs/index.html` and `outputs/match_centre.html`).
* **Docker:** `docker compose up --build`, then open http://localhost:8080
  (optional local LLM for the chatbot: `docker compose --profile llm up --build`).

Every step is deterministic: checksums, pinned libraries and fixed random seeds give the same numbers on every run.